# RF-DETR two-stage, TILED end to end

**Stage 1** — real flat-bug images, 1 class, **tiled**.
**Stage 2** — synthetic 23-class **tiles**.
**Eval** — tiled inference on full trays + merge.

Why tile Stage 1 too: the flat-bug images reach **13,514 px**, and a single 800px pass destroys the small insects — measured directly, flatbug's recall went **0.176 → 0.785** purely by switching to tiled inference, and RF-DETR gained **+0.163 mAP@75** the same way. A prior learned on squashed images is a prior for the wrong scale. Tiling both stages keeps everything at **1:1**.

**Prerequisites:** `flatbug_coco/` (polygon-aware converter) and `insect_dataset_coco_20000_tiled/` (Step 1 of `train_rfdetr_tiled.ipynb`).

> `MAX_TILES_PER_IMAGE = 40` caps the explosion — a 13.5k px image would otherwise > produce ~500 tiles. Empty tiles are dropped before the cap is applied.


In [ ]:
# ============================================================
# SEPTEMBER RUN — RF-DETR two-stage, 1536 px tiles fed 1:1
# ============================================================
# TWO-STAGE: stage 1 pretrains on the REAL flat-bug photographs as a 1-class insect
# localiser; stage 2 fine-tunes that on the mixed 22-class trays. Roughly double the
# GPU time of a single-stage run.
#
# Its premise is worth re-checking before you spend that. The approach exists because
# flatbug beat RF-DETR on localisation (mAP@75 0.468 vs 0.340) — but that was measured
# against ground-truth boxes 1.295x too large on a side, and mAP@75 is exactly the
# metric a systematic box-size bias distorts. If the September tables show the two
# architectures close on mAP@75, the gap this was built to close does not exist.
#
# Everything else matches the other September notebooks: 22 classes, trays at
# 3473x2769 (10.85 px/mm, same as real), boxes that are the insect rather than the
# rotated canvas, and evaluation on 3dtrab_coco_TESTSPLIT.json.

# ============================================================
# CONFIG — two-stage, TILED end to end
# ============================================================
import os, json
GPU_ID = "1"   # two-stage runs after the first four, so 0/1 are free
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
from pathlib import Path

BASE_DIR   = Path(os.environ.get("MA_BASE", "/data/MA_Atul_Ares"))

# Stage 1 — REAL flat-bug data (1 class). Tiled, because these images reach 13,514px
# and a single 800px pass destroys the small insects.
FB_COCO        = BASE_DIR / "flatbug_coco"            # built by the polygon-aware converter
FB_COCO_TILED  = BASE_DIR / "flatbug_coco_tiled"
STAGE1_OUT     = str(BASE_DIR / "rfdetr_output_tiled_stage1_flatbug")
STAGE1_EPOCHS  = 30

# Stage 2 — synthetic 23-class tiles (reuse the tiles from train_rfdetr_tiled.ipynb)
TILED_COCO = BASE_DIR / "insect_dataset_mixed_tiled1536"   # from train_rfdetr_tiled_fullres
SRC_COCO   = BASE_DIR / "insect_dataset_mixed"        # FULL trays, for the final eval
OUTPUT_DIR = str(BASE_DIR / "sept_run" / "rfdetr_twostage_tiled_1536")

# tiling — TILE == RESOLUTION so tiles are fed 1:1
TILE, OVERLAP = 1536, 256   # == RESOLUTION, so tiles are fed 1:1 with NO resize
MIN_VIS, MIN_SIZE = 0.85, 4
MAX_TILES_PER_IMAGE = 40      # cap: a 13.5k px image would otherwise yield ~500 tiles
MERGE_IOU, EDGE_MARGIN, EDGE_FACTOR = 0.6, 16, 0.85
INFER_CONF = 0.05

MODEL_VARIANT, RESOLUTION = "RFDETRLarge", 1536
EPOCHS, BATCH_SIZE, GRAD_ACCUM_STEPS = 30, 4, 4
LR, LR_ENCODER, LR_DROP, WARMUP_EPOCHS = 1e-4, 1.5e-4, 25, 5
WEIGHT_DECAY, MULTI_SCALE, USE_EMA = 1e-4, True, False
CHECKPOINT_INTERVAL, EVAL_INTERVAL, RUN_TEST = 5, 1, True
EARLY_STOPPING, EARLY_STOPPING_PATIENCE, EARLY_STOPPING_MIN_DELTA = True, 10, 0.001
TENSORBOARD, WANDB, NUM_WORKERS, PROGRESS_BAR = True, False, 8, "rich"
NUM_GPUS, DDP_STRATEGY = 1, "auto"

CLASS_NAMES = [
    # 22 classes: Siphonaptera and Raphidioptera dropped;
    # Diplopoda -> Polyxenida, Chilopoda -> Lithobiomorpha + Scutigeromorpha.
    "Lepidoptera",
    "Psocodea",
    "Odonata",
    "Zoraptera",
    "Phasmatodea",
    "Zygentoma",
    "Thysanoptera",
    "Strepsiptera",
    "Archaeognatha",
    "Dermaptera",
    "Diptera",
    "Coleoptera",
    "Hymenoptera",
    "Hemiptera",
    "Orthoptera",
    "Trichoptera",
    "Neuroptera",
    "Araneae",
    "Opiliones",
    "Polyxenida",
    "Lithobiomorpha",
    "Scutigeromorpha",
]
assert (FB_COCO/"train"/"_annotations.coco.json").exists(), f"{FB_COCO} missing - run the converter"
assert (TILED_COCO/"train"/"_annotations.coco.json").exists(), (
    f"{TILED_COCO} missing - run Step 1 of train_rfdetr_tiled.ipynb first")
print(f"GPU {GPU_ID} | tiles {TILE}px @ res {RESOLUTION} (1:1)")
print(f"Stage 1: {FB_COCO} -> {FB_COCO_TILED} -> {STAGE1_OUT}")
print(f"Stage 2: {TILED_COCO} -> {OUTPUT_DIR}")


In [ ]:
# ── Tiling core (grid, box tiling, NMS, edge penalty, merge) ──
import json, math
import numpy as np
from pathlib import Path


def tile_grid(W, H, tile, overlap):
    """Top-left corners covering WxH with `tile` windows and `overlap` px overlap.
    Last row/col is flush to the edge so nothing is missed."""
    step = max(1, tile - overlap)
    xs = list(range(0, max(1, W - tile + 1), step))
    ys = list(range(0, max(1, H - tile + 1), step))
    if not xs or xs[-1] != max(0, W - tile): xs.append(max(0, W - tile))
    if not ys or ys[-1] != max(0, H - tile): ys.append(max(0, H - tile))
    return sorted(set(xs)), sorted(set(ys))


def tile_boxes(boxes, x0, y0, tile, W, H, min_vis=0.85, min_size=4):
    """Clip xyxy boxes into the tile at (x0,y0). Returns (kept_boxes_local, kept_idx).
    A box is kept only if >= min_vis of its area survives the crop, so partial
    insects at tile edges are not learned as whole objects (they are complete in
    the overlapping neighbour tile)."""
    if len(boxes) == 0:
        return np.zeros((0, 4)), []
    b = np.asarray(boxes, float)
    x1 = np.clip(b[:, 0], x0, x0 + tile); y1 = np.clip(b[:, 1], y0, y0 + tile)
    x2 = np.clip(b[:, 2], x0, x0 + tile); y2 = np.clip(b[:, 3], y0, y0 + tile)
    w, h = np.clip(x2 - x1, 0, None), np.clip(y2 - y1, 0, None)
    area_clip = w * h
    area_orig = np.clip((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]), 1e-9, None)
    keep = (area_clip / area_orig >= min_vis) & (w >= min_size) & (h >= min_size)
    idx = np.where(keep)[0]
    out = np.stack([x1[idx] - x0, y1[idx] - y0, x2[idx] - x0, y2[idx] - y0], 1) if len(idx) else np.zeros((0, 4))
    return out, idx.tolist()


def nms(boxes, scores, iou_thr=0.5):
    """Plain greedy NMS. boxes xyxy."""
    if len(boxes) == 0:
        return []
    b = np.asarray(boxes, float); s = np.asarray(scores, float)
    x1, y1, x2, y2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    area = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    order = s.argsort()[::-1]; keep = []
    while order.size:
        i = order[0]; keep.append(int(i))
        if order.size == 1: break
        xx1 = np.maximum(x1[i], x1[order[1:]]); yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]]); yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
        iou = inter / np.clip(area[i] + area[order[1:]] - inter, 1e-9, None)
        order = order[1:][iou <= iou_thr]
    return keep


def edge_penalty(local_boxes, tile, margin=16, factor=0.85):
    """Multiplier that demotes detections touching a tile border. The same object
    is usually complete in the overlapping neighbour tile, so NMS should prefer
    that copy. Mirrors flatbug's EDGE_CASE_MARGIN idea."""
    if len(local_boxes) == 0:
        return np.zeros(0)
    b = np.asarray(local_boxes, float)
    touch = ((b[:, 0] <= margin) | (b[:, 1] <= margin) |
             (b[:, 2] >= tile - margin) | (b[:, 3] >= tile - margin))
    return np.where(touch, factor, 1.0)


def merge_tiles(preds, iou_thr=0.5, per_class=False):
    """preds: list of (boxes_global Nx4, scores N, classes N) from each tile.
    -> merged (boxes, scores, classes) after NMS in full-image coordinates."""
    B = [p[0] for p in preds if len(p[0])]
    if not B:
        return np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)
    boxes = np.concatenate(B)
    scores = np.concatenate([p[1] for p in preds if len(p[0])])
    classes = np.concatenate([p[2] for p in preds if len(p[0])]).astype(int)
    keep_all = []
    for c in (np.unique(classes) if per_class else [None]):
        m = np.where(classes == c)[0] if c is not None else np.arange(len(boxes))
        k = nms(boxes[m], scores[m], iou_thr)
        keep_all += m[k].tolist()
    keep_all = np.array(sorted(keep_all), int)
    return boxes[keep_all], scores[keep_all], classes[keep_all]


## Step 1 — tile the real flat-bug data


In [ ]:
# ── STEP 1 — tile the REAL flat-bug COCO for Stage 1 (run once) ──
import json, numpy as np
from PIL import Image
from tqdm.auto import tqdm

FORCE = False
def export_tiles(src_root, dst_root, splits=("train","valid"), cap=MAX_TILES_PER_IMAGE):
    rng = np.random.default_rng(0)
    for split in splits:
        sd = src_root/split
        if not (sd/"_annotations.coco.json").exists(): continue
        ann = json.load(open(sd/"_annotations.coco.json"))
        cats = sorted(ann["categories"], key=lambda c: c["id"])
        by = {}
        for a in ann["annotations"]: by.setdefault(a["image_id"], []).append(a)
        dd = dst_root/split; dd.mkdir(parents=True, exist_ok=True)
        images, annos, iid, aid = [], [], 1, 1
        for im in tqdm(ann["images"], desc=f"tiling {split}"):
            p = sd/im["file_name"]
            if not p.exists(): continue
            W, H = im["width"], im["height"]
            recs = by.get(im["id"], [])
            boxes = np.array([[a["bbox"][0], a["bbox"][1], a["bbox"][0]+a["bbox"][2],
                               a["bbox"][1]+a["bbox"][3]] for a in recs], float).reshape(-1,4)
            cids = [a["category_id"] for a in recs]
            xs, ys = tile_grid(W, H, TILE, OVERLAP)
            grid = [(x0,y0) for y0 in ys for x0 in xs]
            keep = [(x0,y0,*tile_boxes(boxes,x0,y0,TILE,W,H,MIN_VIS,MIN_SIZE)) for x0,y0 in grid]
            keep = [k for k in keep if len(k[3])]              # drop empty tiles
            if cap and len(keep) > cap:                        # subsample huge images
                keep = [keep[i] for i in rng.choice(len(keep), cap, replace=False)]
            if not keep: continue
            img = Image.open(p).convert("RGB")
            for x0, y0, lb, idx in keep:
                name = f"{Path(im['file_name']).stem}_x{x0}_y{y0}.jpg"
                img.crop((x0,y0,min(x0+TILE,W),min(y0+TILE,H))).save(dd/name,"JPEG",quality=92)
                images.append({"id":iid,"file_name":name,
                               "width":min(TILE,W-x0),"height":min(TILE,H-y0)})
                for b,j in zip(lb, idx):
                    annos.append({"id":aid,"image_id":iid,"category_id":cids[j],
                                  "bbox":[float(b[0]),float(b[1]),float(b[2]-b[0]),float(b[3]-b[1])],
                                  "area":float((b[2]-b[0])*(b[3]-b[1])),"iscrowd":0}); aid += 1
                iid += 1
        json.dump({"info":{"description":f"tiled {split}"},"categories":cats,
                   "images":images,"annotations":annos}, open(dd/"_annotations.coco.json","w"))
        print(f"  {split:5s}: {len(images):7d} tiles, {len(annos):8d} boxes")

if FB_COCO_TILED.exists() and not FORCE and (FB_COCO_TILED/"train"/"_annotations.coco.json").exists():
    print(f"{FB_COCO_TILED} exists - skipping (FORCE=True to rebuild)")
else:
    export_tiles(FB_COCO, FB_COCO_TILED)

_a = json.load(open(FB_COCO_TILED/"train"/"_annotations.coco.json"))
_wh = {i["id"]:(i["width"],i["height"]) for i in _a["images"]}
_fr = np.array([[b["bbox"][2]/_wh[b["image_id"]][0], b["bbox"][3]/_wh[b["image_id"]][1]]
                for b in _a["annotations"]])
print(f"\ncategories: {[c['name'] for c in _a['categories']]}")
print(f"median box: {np.median(_fr[:,0])*100:.1f}% x {np.median(_fr[:,1])*100:.1f}% of tile")
assert np.median(_fr) < 0.25, "boxes look wrong - do NOT train"
print("OK")


## Step 2 — Stage 1: real-image prior at 1:1


In [ ]:
# ── STAGE 1 — pretrain on TILED real flat-bug images (1 class) ──
import sys
from pathlib import Path
_src = str(BASE_DIR/"rf-detr-1.7.1"/"src")
if _src not in sys.path: sys.path.insert(0, _src)
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge
ModelClass = {"RFDETRNano":RFDETRNano,"RFDETRSmall":RFDETRSmall,
              "RFDETRMedium":RFDETRMedium,"RFDETRLarge":RFDETRLarge}[MODEL_VARIANT]

m1 = ModelClass(resolution=RESOLUTION, num_classes=1)   # explicit: 1-class prior
assert m1.model_config.num_classes == 1
print(f"Stage 1: num_classes={m1.model_config.num_classes}, resolution={RESOLUTION} == TILE {TILE} (1:1)")
m1.train(
    dataset_dir=str(FB_COCO_TILED), output_dir=STAGE1_OUT, class_names=["insect"],
    epochs=STAGE1_EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LR, lr_encoder=LR_ENCODER, lr_drop=STAGE1_EPOCHS, warmup_epochs=WARMUP_EPOCHS,
    weight_decay=WEIGHT_DECAY, multi_scale=MULTI_SCALE, use_ema=USE_EMA,
    checkpoint_interval=CHECKPOINT_INTERVAL, eval_interval=EVAL_INTERVAL, run_test=False,
    early_stopping=EARLY_STOPPING, early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, progress_bar=PROGRESS_BAR,
    tensorboard=TENSORBOARD, wandb=WANDB, num_workers=NUM_WORKERS,
    devices=NUM_GPUS, strategy=DDP_STRATEGY, resume=None,
)
STAGE1_CKPT = str(Path(STAGE1_OUT)/"checkpoint_best_total.pth")
assert Path(STAGE1_CKPT).exists(), STAGE1_CKPT
print(f"\nStage-1 (tiled, real) prior: {STAGE1_CKPT}")


## Step 3 — Stage 2: synthetic tiles


In [ ]:
# ── STAGE 2 — fine-tune on the synthetic 23-class TILES ──
# num_classes MUST be explicit: otherwise the loader auto-aligns to the Stage-1
# checkpoint (1 class) and the 23-class labels index out of bounds -> CUDA assert.
from pathlib import Path
STAGE1_CKPT = str(Path(STAGE1_OUT)/"checkpoint_best_total.pth")
assert Path(STAGE1_CKPT).exists(), f"run Stage 1 first: {STAGE1_CKPT}"

model = ModelClass(resolution=RESOLUTION, pretrain_weights=STAGE1_CKPT,
                   num_classes=len(CLASS_NAMES))
assert model.model_config.num_classes == len(CLASS_NAMES), model.model_config.num_classes
print(f"Stage 2: num_classes={model.model_config.num_classes} (head -> {len(CLASS_NAMES)}+1)")
model.train(
    dataset_dir=str(TILED_COCO), output_dir=OUTPUT_DIR, class_names=CLASS_NAMES,
    epochs=EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LR, lr_encoder=LR_ENCODER, lr_drop=LR_DROP, warmup_epochs=WARMUP_EPOCHS,
    weight_decay=WEIGHT_DECAY, multi_scale=MULTI_SCALE, use_ema=USE_EMA,
    checkpoint_interval=CHECKPOINT_INTERVAL, eval_interval=EVAL_INTERVAL, run_test=RUN_TEST,
    early_stopping=EARLY_STOPPING, early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, progress_bar=PROGRESS_BAR,
    tensorboard=TENSORBOARD, wandb=WANDB, num_workers=NUM_WORKERS,
    devices=NUM_GPUS, strategy=DDP_STRATEGY, resume=None,
)
print(f"\nStage-2 checkpoints -> {OUTPUT_DIR}")
print("The test mAP above is on TILES. Run the cells below for the comparable full-tray number.")


## Step 4 — tiled inference on full trays, then merge

**RESUMING after training?** Run only: **CONFIG** -> **tiling core** -> this cell -> the scoring cell. Skip Step 1 (the tiled dataset already exists) and Step 3 (training is done); this cell loads `checkpoint_best_total.pth` from `OUTPUT_DIR` itself.

The test mAP printed by training is on **tiles** and is not comparable to your other runs. This is.


In [ ]:
# ============================================================
# STEP 4 — tiled inference on FULL trays + merge  (comparable to the other runs)
# ============================================================
import sys, json, time
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

# self-contained: rfdetr lives in the source tree, not site-packages, so the path
# insert must happen here too (the training cell above may be skipped on a resume)
_src = str(Path(".").resolve() / "rf-detr-1.7.1" / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)
from rfdetr import RFDETR

EVAL_SPLIT = "test"
CKPT = Path(OUTPUT_DIR) / "checkpoint_best_total.pth"
assert CKPT.exists(), f"missing {CKPT}"
m = RFDETR.from_checkpoint(str(CKPT), resolution=RESOLUTION)  # from_checkpoint does NOT restore resolution (falls back to the 704 default)

src_dir = SRC_COCO / EVAL_SPLIT
ann = json.load(open(src_dir / "_annotations.coco.json"))
cats = sorted(ann["categories"], key=lambda c: c["id"])
name_to_idx = {c["name"]: i for i, c in enumerate(cats)}
catid_to_idx = {c["id"]: i for i, c in enumerate(cats)}
GT = {}
for a in ann["annotations"]:
    b = a["bbox"]
    GT.setdefault(a["image_id"], []).append(
        (catid_to_idx[a["category_id"]], b[0], b[1], b[0]+b[2], b[1]+b[3]))

MAX_EVAL_IMAGES = None          # None = whole split
metas = ann["images"] if MAX_EVAL_IMAGES is None else ann["images"][:MAX_EVAL_IMAGES]

preds_out, t0 = {}, time.time()
for im in tqdm(metas, desc="tiled inference"):
    p = src_dir / im["file_name"]
    if not p.exists(): continue
    W, H = im["width"], im["height"]
    img = Image.open(p).convert("RGB")
    xs, ys = tile_grid(W, H, TILE, OVERLAP)
    tile_preds = []
    for y0 in ys:
        for x0 in xs:
            crop = img.crop((x0, y0, min(x0+TILE, W), min(y0+TILE, H)))
            d = m.predict(crop, threshold=INFER_CONF)
            if d is None or len(d) == 0: continue
            lb = np.asarray(d.xyxy, float).reshape(-1, 4)
            sc = np.asarray(d.confidence, float)
            nm = d.data.get("class_name", None)
            cl = (np.array([name_to_idx.get(str(n), 0) for n in nm]) if nm is not None
                  and len(nm) == len(sc) else np.asarray(d.class_id, int))
            sc = sc * edge_penalty(lb, TILE, EDGE_MARGIN, EDGE_FACTOR)
            tile_preds.append((lb + np.array([x0, y0, x0, y0]), sc, cl))
    # CLASS-AGNOSTIC. per_class=True let one insect, detected as two different classes
    # in two overlapping tiles, survive as two boxes. Measured on the August runs:
    # 20% of this model's detections were co-located pairs with different labels.
    mb, ms, mc = merge_tiles(tile_preds, iou_thr=MERGE_IOU, per_class=False)
    preds_out[im["id"]] = {"boxes": mb.tolist(), "scores": ms.tolist(), "classes": mc.tolist()}

json.dump(preds_out, open(Path(OUTPUT_DIR) / f"tiled_preds_{EVAL_SPLIT}.json", "w"))
print(f"\n{sum(len(v['scores']) for v in preds_out.values())} merged detections over "
      f"{len(preds_out)} full trays in {time.time()-t0:.0f}s")


In [ ]:
# ── STEP 5 — score the merged full-tray predictions (COCO 101-pt AP) ──
import json, numpy as np, pandas as pd
from pathlib import Path

# ── self-contained: rebuild GT / CLASS_NAMES from the COCO annotations when the
# inference cell above was skipped (e.g. re-scoring an already-finished run) ──
EVAL_SPLIT = globals().get("EVAL_SPLIT", "test")
if "GT" not in globals() or "CLASS_NAMES" not in globals():
    _ann = json.load(open(Path(SRC_COCO)/EVAL_SPLIT/"_annotations.coco.json"))
    _cats = sorted(_ann["categories"], key=lambda c: c["id"])
    CLASS_NAMES = [c["name"] for c in _cats]
    _c2i = {c["id"]: i for i, c in enumerate(_cats)}
    GT = {}
    for _a in _ann["annotations"]:
        _b = _a["bbox"]
        GT.setdefault(_a["image_id"], []).append(
            (_c2i[_a["category_id"]], _b[0], _b[1], _b[0]+_b[2], _b[1]+_b[3]))
    print(f"rebuilt GT: {len(GT)} images, {len(_ann['annotations'])} boxes, "
          f"{len(CLASS_NAMES)} classes (split={EVAL_SPLIT})")
IOU_VEC = np.linspace(0.5, 0.95, 10)
NC = len(CLASS_NAMES)

def iou_mat(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    a, b = np.asarray(a, float), np.asarray(b, float)
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb-lt, 0, None); inter = wh[..., 0]*wh[..., 1]
    aa = (a[:, 2]-a[:, 0])*(a[:, 3]-a[:, 1]); ab = (b[:, 2]-b[:, 0])*(b[:, 3]-b[:, 1])
    return inter/np.clip(aa[:, None]+ab[None]-inter, 1e-9, None)

# NumPy 2.0 removed np.trapz in favour of np.trapezoid. The training envs run
# different NumPy majors (flatbug torch 2.13 / rfdetr torch 2.9), so bind once
# and use _TRAPZ rather than depending on which one is installed.
_TRAPZ = getattr(np, "trapezoid", None) or np.trapz
def _ap(rec, prec):
    mrec = np.concatenate(([0.], rec, [1.])); mpre = np.concatenate(([1.], prec, [0.]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101); return float(_TRAPZ(np.interp(x, mrec, mpre), x))

if "preds_out" not in globals():          # re-score without re-running inference
    _f = Path(OUTPUT_DIR)/f"tiled_preds_{EVAL_SPLIT}.json"
    preds_out = {int(k): v for k, v in json.load(open(_f)).items()}
    print(f"loaded {len(preds_out)} images from {_f}")

tps, confs, pcls, tcls = [], [], [], []
for iid, pr in preds_out.items():
    g = GT.get(iid, [])
    gb = np.array([x[1:] for x in g]).reshape(-1, 4); gc = np.array([x[0] for x in g], int)
    pb = np.array(pr["boxes"]).reshape(-1, 4); ps = np.array(pr["scores"]); pc = np.array(pr["classes"], int)
    tp = np.zeros((len(pb), len(IOU_VEC)), bool)
    if len(pb) and len(gb):
        M = iou_mat(pb, gb); same = pc[:, None] == gc[None, :]
        for ti, thr in enumerate(IOU_VEC):
            ci, gi = np.where((M >= thr) & same)
            if not len(ci): continue
            o = np.argsort(-M[ci, gi]); up, ug = set(), set()
            for c, gg in zip(ci[o], gi[o]):
                if c in up or gg in ug: continue
                up.add(c); ug.add(gg); tp[c, ti] = True
    tps.append(tp); confs.append(ps); pcls.append(pc); tcls.append(gc)

tp = np.concatenate(tps); conf = np.concatenate(confs)
pcls = np.concatenate(pcls); tcls = np.concatenate(tcls)
o = np.argsort(-conf); tp, conf, pcls = tp[o], conf[o], pcls[o]
n_gt = np.bincount(tcls, minlength=NC)

rows, ap50, ap75, ap5095 = [], [], [], []
precs, recs = [], []
for c in range(NC):
    i = pcls == c
    if n_gt[c] == 0 or i.sum() == 0:
        rows.append({"class": CLASS_NAMES[c], "instances": int(n_gt[c]), "AP50": 0.0, "AP75": 0.0,
                     "AP50_95": 0.0, "precision": 0.0, "recall": 0.0, "F1": 0.0, "conf_at_bestF1": 0.0})
        ap50.append(0.0); ap75.append(0.0); ap5095.append(0.0)
        precs.append(0.0); recs.append(0.0); continue
    tpc = tp[i].cumsum(0); fpc = (1-tp[i]).cumsum(0)
    rec = tpc/(n_gt[c]+1e-16); prec = tpc/(tpc+fpc+1e-16)
    aps = [_ap(rec[:, t], prec[:, t]) for t in range(len(IOU_VEC))]
    ap50.append(aps[0]); ap75.append(aps[5]); ap5095.append(float(np.mean(aps)))
    # Precision and recall at the best-F1 operating point, IoU 0.5 — the same
    # convention Ultralytics reports, so the tiled and whole-tray rows of the
    # comparison are directly comparable. These curves were already being computed
    # and thrown away after integrating AP.
    _p50, _r50 = prec[:, 0], rec[:, 0]
    _f1 = 2*_p50*_r50/np.clip(_p50+_r50, 1e-16, None)
    _b = int(np.argmax(_f1))
    precs.append(float(_p50[_b])); recs.append(float(_r50[_b]))
    rows.append({"class": CLASS_NAMES[c], "instances": int(n_gt[c]),
                 "AP50": round(aps[0], 4), "AP75": round(aps[5], 4),
                 "AP50_95": round(float(np.mean(aps)), 4),
                 "precision": round(float(_p50[_b]), 4),
                 "recall": round(float(_r50[_b]), 4),
                 "F1": round(float(_f1[_b]), 4),
                 "conf_at_bestF1": round(float(conf[i][_b]), 4)})
df = pd.DataFrame(rows)
df.to_csv(Path(OUTPUT_DIR)/"tiled_per_class.csv", index=False)
print(df.to_string(index=False))
print(f"\n=== TILED RF-DETR, full trays ({EVAL_SPLIT}) ===")
print(f"  mAP@50:95 : {np.mean(ap5095):.4f}")
print(f"  mAP@50    : {np.mean(ap50):.4f}")
print(f"  mAP@75    : {np.mean(ap75):.4f}   <- the metric this experiment is about")
print(f"  precision : {np.mean(precs):.4f}   (class mean at best-F1, IoU 0.5)")
print(f"  recall    : {np.mean(recs):.4f}")
print( "  compare against: RF-DETR 800px single-stage 0.3476 (@50 0.6408, @75 0.3400)")
print( "                   flatbug-L                0.4189 (@50 0.6265, @75 0.4681)")
